# 01 — Classical EDA
Load data, EDA, PCA, UMAP, anomaly detection.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
SEED = 42
np.random.seed(SEED)
for d in ['../outputs/figures','../outputs/tables','../outputs/kernels',
          '../outputs/logs','../data/processed','../data/raw']:
    os.makedirs(d, exist_ok=True)
print("Setup done.")

## Data Loading

In [ ]:
from src.data import download_wine_quality, load_wine_quality, WINE_FEATURE_COLS, WINE_TARGET_COL
download_wine_quality('../data/raw/winequality-red.csv')
df = load_wine_quality('../data/raw/winequality-red.csv')
print(f"Shape: {df.shape}")

## Missing Values & Duplicates

In [ ]:
print("Missing values:\n", df.isnull().sum().to_string())
n_dups = df.duplicated().sum()
print(f"Duplicate rows: {n_dups} (retained, not removed — documented)")

## Descriptive Statistics

In [ ]:
desc = df.describe()
print(desc)
desc.to_csv('../outputs/tables/descriptive_statistics.csv')
print("Saved descriptive_statistics.csv")

## Feature Distributions

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15,12))
axes = axes.flatten()
for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=9)
for j in range(len(df.columns), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature Distributions — UCI Red Wine Quality', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/01_feature_distributions.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved 01_feature_distributions.png")

## Boxplots

In [ ]:
from sklearn.preprocessing import StandardScaler
X = df[WINE_FEATURE_COLS].values
X_scaled = StandardScaler().fit_transform(X)
df_std = pd.DataFrame(X_scaled, columns=WINE_FEATURE_COLS)
fig, ax = plt.subplots(figsize=(14,6))
df_std.boxplot(ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
ax.set_title('Standardized Boxplots')
plt.tight_layout()
plt.savefig('../outputs/figures/02_boxplots.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved 02_boxplots.png")

## Correlation Matrix

In [ ]:
corr = df.corr()
fig, ax = plt.subplots(figsize=(12,10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5, annot_kws={"size":8})
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.savefig('../outputs/figures/03_correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.close()
pairs = corr.unstack().sort_values(key=abs, ascending=False)
pairs = pairs[pairs.index.get_level_values(0) != pairs.index.get_level_values(1)]
print("Top correlations:\n", pairs.head(10).to_string())

## Standardize & Save Processed Data

In [ ]:
df_processed = df.copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[WINE_FEATURE_COLS])
df_processed.to_csv('../data/processed/wine_quality_processed.csv', index=False)
print(f"Saved processed data. Shape: {df_processed.shape}")

## PCA

In [ ]:
from src.classical import run_pca
pca_emb, pca_var = run_pca(X_scaled)
print(f"PCA explained variance: {pca_var.round(4)}")
fig, ax = plt.subplots(figsize=(8,6))
sc = ax.scatter(pca_emb[:,0], pca_emb[:,1], c=df[WINE_TARGET_COL], cmap='viridis', alpha=0.6, s=10)
plt.colorbar(sc, ax=ax, label='Quality')
ax.set_xlabel(f"PC1 ({pca_var[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca_var[1]*100:.1f}%)")
ax.set_title("PCA — Red Wine Quality")
plt.tight_layout()
plt.savefig('../outputs/figures/04_pca.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved 04_pca.png")

## UMAP

In [ ]:
from src.classical import run_umap
print("Running UMAP...")
umap_emb = run_umap(X_scaled, random_state=SEED)
fig, ax = plt.subplots(figsize=(8,6))
sc = ax.scatter(umap_emb[:,0], umap_emb[:,1], c=df[WINE_TARGET_COL], cmap='viridis', alpha=0.6, s=10)
plt.colorbar(sc, ax=ax, label='Quality')
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.set_title("UMAP — Red Wine Quality")
plt.tight_layout()
plt.savefig('../outputs/figures/05_umap.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved 05_umap.png")

## Isolation Forest & LOF

In [ ]:
from src.classical import run_isolation_forest, run_lof
if_scores = run_isolation_forest(X_scaled, random_state=SEED)
lof_scores = run_lof(X_scaled)
print(f"IF scores:  mean={if_scores.mean():.4f}, max={if_scores.max():.4f}")
print(f"LOF scores: mean={lof_scores.mean():.4f}, max={lof_scores.max():.4f}")
fig, axes = plt.subplots(1,2,figsize=(12,5))
for ax, sc_vals, title in zip(axes, [if_scores, lof_scores], ['IsolationForest','LOF']):
    scat = ax.scatter(pca_emb[:,0], pca_emb[:,1], c=sc_vals, cmap='YlOrRd', alpha=0.6, s=10)
    ax.set_title(f"{title} Anomaly Score (PCA)")
    plt.colorbar(scat, ax=ax)
plt.tight_layout()
plt.savefig('../outputs/figures/06_classical_anomalies.png', dpi=120, bbox_inches='tight')
plt.close()
np.save('../outputs/tables/if_scores_full.npy', if_scores)
np.save('../outputs/tables/lof_scores_full.npy', lof_scores)
np.save('../outputs/tables/X_scaled_full.npy', X_scaled)
np.save('../outputs/tables/pca_emb_full.npy', pca_emb)
print("Notebook 01 complete.")